<a href="https://colab.research.google.com/github/avocado-planet/09-RAG/blob/main/09_comprehensive_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 包括的 RAG 実装ガイド — 基礎から本番品質まで

このノートブックでは、RAG (Retrieval-Augmented Generation) を **基礎的な構成からプロダクション品質の構成まで** 段階的に実装します。

**対象ドキュメント**: [Cloud-IAM 公式ドキュメント](https://documentation.cloud-iam.com/) — Keycloak のマネージドサービスに関するドキュメントサイト

## 実装する技術要素

| # | 技術要素 | セクション |
|---|---|---|
| 1 | Document Loader (WebBaseLoader + URL 自動収集) | §01 |
| 2 | RecursiveCharacterTextSplitter (overlap 付き) | §02 |
| 3 | MarkdownHeaderTextSplitter (構造対応) | §02 |
| 4 | OpenAI Embeddings | §03 |
| 5 | Chroma Vector Store + 永続化 | §04 |
| 6 | Similarity Search (基本) | §05 |
| 7 | MMR 検索 (多様性) | §05 |
| 8 | Similarity Score Threshold (しきい値) | §05 |
| 9 | ハイブリッド検索 (BM25 + Vector) | §06 |
| 10 | リランキング (Cohere Rerank) | §07 |
| 11 | Multi-Query Retriever (クエリ展開) | §08 |
| 12 | Parent Document Retriever (親子チャンク) | §09 |
| 13 | 基本 RAG チェーン (LCEL) | §10 |
| 14 | 引用元付き RAG (RunnableParallel) | §11 |
| 15 | ストリーミング出力 | §12 |
| 16 | 検索方式の比較実験 | §13 |

### 全体フロー

```
 Cloud-IAM Docs                          ユーザー質問
 (Web サイト)                                 ↓
     ↓                                   [Embedding]
 [§01] URL収集 + WebBaseLoader                ↓
     ↓                                [§05-09] Retriever
 [§02] Splitter                               ↓
     ↓                            関連チャンク
 [§03] Embedding                              ↓
     ↓                           [§10-11] RAG Chain → 回答
 [§04] Chroma DB ──── 検索 ──→
```

> **前提**: Google Colab で実行。`OPENAI_API_KEY` を Colab シークレットに登録済み。
> リランキング (§07) には追加で `COHERE_API_KEY` が必要 (オプション)。


---
## 00. ライブラリのインストール

本ノートブックで使用する全パッケージを一括インストールします。


In [16]:
!pip install -q langchain langchain-classic langchain-core langchain-openai langchain-community langchain-text-splitters langchain-chroma beautifulsoup4 rank_bm25

# Reranking を試す場合のみ (オプション)
!pip install -q langchain-cohere


---
## 01. API キーの設定と共通インポート


In [2]:
import os
import warnings
from google.colab import userdata

warnings.filterwarnings("ignore")

# --- 必須 ---
api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY が未設定です。Colab の鍵アイコンからシークレットに登録してください。"
    )
os.environ["OPENAI_API_KEY"] = api_key

# --- オプション: リランキング (§07) で使用 ---
try:
    cohere_key = userdata.get("COHERE_API_KEY")
    if cohere_key:
        os.environ["COHERE_API_KEY"] = cohere_key
        print("✅ COHERE_API_KEY をセットしました (リランキング使用可能)")
    else:
        print("ℹ️  COHERE_API_KEY 未設定 → リランキングセクションはスキップ")
except Exception:
    print("ℹ️  COHERE_API_KEY 取得エラー → リランキングセクションはスキップ")

print("✅ OPENAI_API_KEY をセットしました")


ℹ️  COHERE_API_KEY 取得エラー → リランキングセクションはスキップ
✅ OPENAI_API_KEY をセットしました


---
## 02. Document Loader — Cloud-IAM ドキュメントの取得

Cloud-IAM のドキュメントは Web サイト (VitePress 製) として公開されているため、
`GitLoader` ではなく **`WebBaseLoader`** を使います。

### 取得手順
1. トップページの HTML をリクエストし、サイドバーのリンクから **全ページの URL を自動収集**
2. `WebBaseLoader` で各ページのテキストを取得
3. ナビゲーション等のノイズを除去

### ポイント
- `WebBaseLoader` は内部で BeautifulSoup を使い、HTML → クリーンなテキストに変換
- メタデータの `source` にページ URL が入るため、引用元として使える
- カテゴリ情報 (get-started / how-to-guides / references / resources) を URL パスから抽出してメタデータに追加


In [3]:
import requests
from bs4 import BeautifulSoup
from langchain_community.document_loaders import WebBaseLoader

# ============================
# Step 1: 全ページの URL を収集
# ============================
BASE_URL = "https://documentation.cloud-iam.com"

resp = requests.get(BASE_URL)
soup = BeautifulSoup(resp.text, "html.parser")

# サイドバーのリンクから全ページ URL を抽出
urls = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    # 内部リンク (.html で終わるもの) のみ
    if href.startswith("/") and href.endswith(".html"):
        urls.add(BASE_URL + href)
    elif href.startswith(BASE_URL) and ".html" in href:
        urls.add(href.split("#")[0])  # アンカー (#xxx) を除去

urls = sorted(urls)
print(f"取得対象ページ数: {len(urls)}")
for u in urls[:5]:
    print(f"  {u}")
print(f"  ... (他 {len(urls) - 5} ページ)")

# ============================
# Step 2: 全ページを読み込み
# ============================
loader = WebBaseLoader(
    web_paths=urls,
    encoding="utf-8",
)
raw_docs = loader.load()

# ============================
# Step 3: クリーニング + メタデータ強化
# ============================
cleaned_docs = []
for doc in raw_docs:
    content = doc.page_content.strip()
    # ナビゲーション等のノイズでテキストが極端に短い場合を除外
    if len(content) < 100:
        continue

    # URL パスからカテゴリを抽出してメタデータに追加
    source = doc.metadata.get("source", "")
    if "/get-started/" in source:
        category = "get-started"
    elif "/how-to-guides/" in source:
        category = "how-to-guides"
    elif "/references/" in source:
        category = "references"
    elif "/resources/" in source:
        category = "resources"
    else:
        category = "other"
    doc.metadata["category"] = category

    cleaned_docs.append(doc)

raw_docs = cleaned_docs
print(f"\n読み込んだドキュメント数: {len(raw_docs)}")
print(f"カテゴリ別:")
from collections import Counter
cat_counts = Counter(d.metadata["category"] for d in raw_docs)
for cat, cnt in sorted(cat_counts.items()):
    print(f"  {cat}: {cnt} ページ")

print(f"\n--- 最初のドキュメント ---")
print(f"ソース: {raw_docs[0].metadata.get('source')}")
print(f"カテゴリ: {raw_docs[0].metadata.get('category')}")
print(f"本文 (先頭 200 文字): {raw_docs[0].page_content[:200]}...")


取得対象ページ数: 101
  https://documentation.cloud-iam.com/faq/billing.html
  https://documentation.cloud-iam.com/faq/cloud-provider.html
  https://documentation.cloud-iam.com/faq/customization.html
  https://documentation.cloud-iam.com/faq/keycloak.html
  https://documentation.cloud-iam.com/faq/migrate-off.html
  ... (他 96 ページ)

読み込んだドキュメント数: 101
カテゴリ別:
  get-started: 4 ページ
  how-to-guides: 30 ページ
  other: 9 ページ
  references: 29 ページ
  resources: 29 ページ

--- 最初のドキュメント ---
ソース: https://documentation.cloud-iam.com/faq/billing.html
カテゴリ: other
本文 (先頭 200 文字): 




FAQ | Billing | Cloud-IAM | DOCS

















Skip to contentCloud-IAM | DOCSSearchKAppearanceMenuReturn to top Sidebar Navigation Go to cloud-iam.com 🌐Welcome 👋Get startedAccess to Cloud-IAM...


---
## 03. Text Splitter — ドキュメントの分割

2 種類の Splitter を試し、結果を比較します。

### 方法 A: `RecursiveCharacterTextSplitter` (RAG の標準)
- 段落 → 行 → 文 の順で再帰的に分割
- `chunk_overlap` で境界の情報分断を緩和

### 方法 B: `MarkdownHeaderTextSplitter` + Recursive (構造対応)
- Markdown の見出し (`#`, `##`) で先に分割 → 更に Recursive で細分化
- メタデータに見出し情報が付与される → 検索結果で「どの章か」が分かる


In [4]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# ========================================
# 方法 A: RecursiveCharacterTextSplitter
# ========================================
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "。", ".", " ", ""],
)
docs_recursive = recursive_splitter.split_documents(raw_docs)

print(f"=== 方法 A: RecursiveCharacterTextSplitter ===")
print(f"チャンク数: {len(docs_recursive)}")
print(f"先頭チャンク:")
print(f"  メタデータ: {docs_recursive[0].metadata}")
print(f"  本文 (先頭 150 文字): {docs_recursive[0].page_content[:150]}...")


=== 方法 A: RecursiveCharacterTextSplitter ===
チャンク数: 1127
先頭チャンク:
  メタデータ: {'source': 'https://documentation.cloud-iam.com/faq/billing.html', 'title': 'FAQ | Billing | Cloud-IAM | DOCS', 'description': 'Frequently asked questions about Cloud-IAM billing.', 'language': 'en-US', 'category': 'other'}
  本文 (先頭 150 文字): FAQ | Billing | Cloud-IAM | DOCS...


In [5]:
# ========================================
# 方法 B: MarkdownHeaderTextSplitter + Recursive
# ========================================
headers_to_split_on = [
    ("#", "Header_1"),
    ("##", "Header_2"),
    ("###", "Header_3"),
]
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False,  # 見出し行自体もチャンクに含める
)

# まず見出しで分割 (メタデータに見出し情報が付く)
md_docs = []
for doc in raw_docs:
    splits = md_splitter.split_text(doc.page_content)
    for s in splits:
        # 元のメタデータ (source 等) を引き継ぐ
        s.metadata.update(doc.metadata)
    md_docs.extend(splits)

# さらに Recursive で細分化
docs_md_recursive = recursive_splitter.split_documents(md_docs)

print(f"\n=== 方法 B: Markdown Header + Recursive ===")
print(f"見出し分割後: {len(md_docs)} チャンク")
print(f"Recursive 再分割後: {len(docs_md_recursive)} チャンク")
print(f"\n先頭チャンク:")
print(f"  メタデータ: {docs_md_recursive[0].metadata}")
print(f"  本文 (先頭 150 文字): {docs_md_recursive[0].page_content[:150]}...")



=== 方法 B: Markdown Header + Recursive ===
見出し分割後: 104 チャンク
Recursive 再分割後: 1114 チャンク

先頭チャンク:
  メタデータ: {'source': 'https://documentation.cloud-iam.com/faq/billing.html', 'title': 'FAQ | Billing | Cloud-IAM | DOCS', 'description': 'Frequently asked questions about Cloud-IAM billing.', 'language': 'en-US', 'category': 'other'}
  本文 (先頭 150 文字): FAQ | Billing | Cloud-IAM | DOCS...


**以降は方法 A (`docs_recursive`) を使って進めます。**
方法 B に切り替えたい場合は `docs = docs_md_recursive` に変更してください。


In [6]:
# 以降のセクションで使うチャンクを決定
docs = docs_recursive
print(f"使用するチャンク数: {len(docs)}")


使用するチャンク数: 1127


---
## 04. Embeddings — テキストのベクトル化

`text-embedding-3-small` (1536次元) を使います。

### 動作確認
クエリを 1 つベクトル化して、次元数と値の分布を確認します。


In [26]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 動作確認
query = "Cloud-IAM のフリープランの最大ユーザ数は"
vector = embeddings.embed_query(query)

print(f"ベクトル次元数: {len(vector)}")
print(f"先頭 5 要素: {vector[:5]}")

import statistics
print(f"平均: {statistics.mean(vector):.6f}")
print(f"標準偏差: {statistics.stdev(vector):.6f}")
print(f"最小/最大: {min(vector):.4f} / {max(vector):.4f}")


ベクトル次元数: 1536
先頭 5 要素: [-0.009063720703125, 0.048919677734375, 0.053619384765625, 0.00279998779296875, 0.033538818359375]
平均: -0.000152
標準偏差: 0.025520
最小/最大: -0.1014 / 0.1207


---
## 05. Vector Store (Chroma) — ベクトルの保存と永続化

ドキュメントを Embedding して Chroma に保存します。

### 永続化
`persist_directory` を指定すると、ディスクに保存されて再利用可能。
ノートブックを再起動しても再度 Embedding を計算する必要がなくなります。


In [28]:
from langchain_chroma import Chroma
import shutil, os

PERSIST_DIR = "./chroma_db"

# 前回のデータがあれば削除 (クリーン再作成)
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

# ドキュメントを Embedding して Chroma に保存 (永続化)
db = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory=PERSIST_DIR,
)
print(f"✅ Chroma DB 作成完了: {db._collection.count()} チャンク")
print(f"   永続化先: {PERSIST_DIR}")


InternalError: Query error: Database error: error returned from database: (code: 1032) attempt to write a readonly database

In [ ]:
# === 永続化された DB を再読み込みする方法 ===
# (参考: 次回以降の実行時に使う)
# db = Chroma(
#     persist_directory=PERSIST_DIR,
#     embedding_function=embeddings,
# )
# print(f"再読み込み完了: {db._collection.count()} チャンク")


---
## 06. 検索方式 ① — Similarity / MMR / Threshold の比較

同じクエリで 3 つの検索方式を試し、結果の違いを比較します。

| 方式 | 特徴 |
|---|---|
| **Similarity** | コサイン類似度の上位 k 件を返す (最も基本) |
| **MMR** | 類似度 + 多様性を両立させる |
| **Threshold** | 類似度スコアが閾値以上のものだけ返す |


In [25]:
query = "Cloud-IAM のフリープランの最大ユーザ数は?"

# ========================================
# (1) Similarity Search — 最も基本的な検索
# ========================================
retriever_sim = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
results_sim = retriever_sim.invoke(query)

print("=" * 60)
print("(1) Similarity Search (k=4)")
print("=" * 60)
for i, d in enumerate(results_sim, 1):
    print(f"  [{i}] {d.metadata.get('source', '?')}")
    print(f"      {d.page_content[:100]}...")


(1) Similarity Search (k=4)
  [1] https://documentation.cloud-iam.com/faq/users-management.html
      .000.Therefore, a plan that accommodates more than 49.000 users, such as the 50k Plan with up to 50k...
  [2] https://documentation.cloud-iam.com/references/plans.html
      .Visit Official SitePlans ​At Cloud-IAM, our plans are tailored to your needs, based on the total nu...
  [3] https://documentation.cloud-iam.com/references/architecture-insights.html
      Cloud-IAM Architecture Insights | Cloud-IAM | DOCS...
  [4] https://documentation.cloud-iam.com/faq/users-management.html
      FAQ | Users Management | Cloud-IAM | DOCS...


In [10]:
# ========================================
# (2) MMR — 多様性を重視
# ========================================
retriever_mmr = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
)
results_mmr = retriever_mmr.invoke(query)

print("=" * 60)
print("(2) MMR Search (k=4, fetch_k=20, λ=0.5)")
print("=" * 60)
for i, d in enumerate(results_mmr, 1):
    print(f"  [{i}] {d.metadata.get('source', '?')}")
    print(f"      {d.page_content[:100]}...")


(2) MMR Search (k=4, fetch_k=20, λ=0.5)
  [1] https://documentation.cloud-iam.com/references/architecture-insights.html
      Cloud-IAM Architecture Insights | Cloud-IAM | DOCS...
  [2] https://documentation.cloud-iam.com/references/deployment-maintenance.html
      Deployment Maintenance | Cloud-IAM | DOCS...
  [3] https://documentation.cloud-iam.com/get-started/deploy-my-keycloak.html
      .Cloud-IAM does not read, use, or process any user information stored in your Keycloak cluster.Two w...
  [4] https://documentation.cloud-iam.com/resources/multitenant-with-keycloak.html
      Handle multitenant organization on Keycloak | Cloud-IAM | DOCS...


In [11]:
# ========================================
# (3) Similarity Score Threshold
# ========================================
retriever_thresh = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 10, "score_threshold": 0.3},
)
results_thresh = retriever_thresh.invoke(query)

print("=" * 60)
print(f"(3) Score Threshold (>= 0.3): {len(results_thresh)} 件ヒット")
print("=" * 60)
for i, d in enumerate(results_thresh, 1):
    print(f"  [{i}] {d.metadata.get('source', '?')}")
    print(f"      {d.page_content[:100]}...")

# --- 比較サマリー ---
print("\n" + "=" * 60)
print("比較サマリー")
print("=" * 60)
sim_sources = [d.metadata.get("source") for d in results_sim]
mmr_sources = [d.metadata.get("source") for d in results_mmr]
print(f"  Similarity ソース: {sim_sources}")
print(f"  MMR ソース:        {mmr_sources}")
print(f"  Threshold 件数:    {len(results_thresh)}")
overlap = set(sim_sources) & set(mmr_sources)
print(f"  Similarity ∩ MMR:  {overlap}")
print(f"  → MMR は {'より多様な' if len(overlap) < len(sim_sources) else '同じ'} 結果を返しています")


(3) Score Threshold (>= 0.3): 10 件ヒット
  [1] https://documentation.cloud-iam.com/references/architecture-insights.html
      Cloud-IAM Architecture Insights | Cloud-IAM | DOCS...
  [2] https://documentation.cloud-iam.com/references/incident-management.html
      Cloud-IAM Incident Management | Cloud-IAM | DOCS...
  [3] https://documentation.cloud-iam.com/how-to-guides/extensions-management.html
      Extensions Management | Cloud-IAM | DOCS...
  [4] https://documentation.cloud-iam.com/faq/product.html
      FAQ | Product | Cloud-IAM | DOCS...
  [5] https://documentation.cloud-iam.com/references/plans.html
      Plans | Cloud-IAM | DOCS...
  [6] https://documentation.cloud-iam.com/faq/cloud-provider.html
      FAQ | Cloud Providers | Cloud-IAM | DOCS...
  [7] https://documentation.cloud-iam.com/references/organization-role.html
      Organization of Keycloak deployment managed by Cloud-IAM | Cloud-IAM | DOCS...
  [8] https://documentation.cloud-iam.com/references/deployment-maintenance.h

---
## 07. 検索方式 ② — ハイブリッド検索 (BM25 + Vector)

**ベクトル検索** と **キーワード検索 (BM25)** を組み合わせる方式です。

### なぜ組み合わせるのか
- ベクトル検索: 意味的な類似性に強い (「車」→「自動車」がヒット)
- BM25: 固有名詞・型番・エラーコードに強い (「EC2」「gpt-4o-mini」等)
- → **両方の良いとこ取り**

内部では **RRF (Reciprocal Rank Fusion)** で結果を統合します。


In [24]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# BM25 Retriever (キーワード検索)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 4

# Vector Retriever
vector_retriever = db.as_retriever(search_kwargs={"k": 4})

# ハイブリッド Retriever (BM25: 40%, Vector: 60%)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],
)

# --- 3 方式を比較 ---
queries = [
    "Cloud-IAM のフリープランの最大ユーザ数は?",             # 意味検索向き
    "mTLS CRL",                                     # キーワード検索向き (技術用語)
]

for q in queries:
    print("=" * 60)
    print(f"クエリ: {q}")
    print("=" * 60)

    res_vec = vector_retriever.invoke(q)
    res_bm25 = bm25_retriever.invoke(q)
    res_hybrid = hybrid_retriever.invoke(q)

    print(f"  Vector:  {[d.metadata.get('source') for d in res_vec[:3]]}")
    print(f"  BM25:    {[d.metadata.get('source') for d in res_bm25[:3]]}")
    print(f"  Hybrid:  {[d.metadata.get('source') for d in res_hybrid[:3]]}")
    print()


クエリ: Cloud-IAM のフリープランの最大ユーザ数は?
  Vector:  ['https://documentation.cloud-iam.com/faq/users-management.html', 'https://documentation.cloud-iam.com/references/plans.html', 'https://documentation.cloud-iam.com/references/architecture-insights.html']
  BM25:    ['https://documentation.cloud-iam.com/references/security.html', 'https://documentation.cloud-iam.com/get-started/migrate-to-cloud-iam.html', 'https://documentation.cloud-iam.com/references/incident-management.html']
  Hybrid:  ['https://documentation.cloud-iam.com/references/architecture-insights.html', 'https://documentation.cloud-iam.com/faq/users-management.html', 'https://documentation.cloud-iam.com/references/plans.html']

クエリ: mTLS CRL
  Vector:  ['https://documentation.cloud-iam.com/how-to-guides/configure-mTLS.html', 'https://documentation.cloud-iam.com/how-to-guides/configure-mTLS.html', 'https://documentation.cloud-iam.com/how-to-guides/configure-mTLS.html']
  BM25:    ['https://documentation.cloud-iam.com/how-to-guides/c

---
## 08. 検索方式 ③ — リランキング (Cohere Rerank)

**2 段構えの検索**: Retriever が広めに候補を取り → Reranker が精密に再評価。

- Cross-Encoder はクエリとドキュメントを **同時に読んで** 関連度をスコア付け
- Embedding (Bi-Encoder) より **本質的に精度が高い**
- **プロダクション RAG の定番構成**

> ⚠️ `COHERE_API_KEY` が未設定の場合、このセクションはスキップされます。


In [18]:
if os.environ.get("COHERE_API_KEY"):
    try:
        !pip install -q langchain-cohere
        from langchain_classic.retrievers import ContextualCompressionRetriever
        from langchain_cohere import CohereRerank

        # 1段目: 広めに取る (TOP-15)
        base_retriever = db.as_retriever(search_kwargs={"k": 15})

        # 2段目: Reranker で精密に TOP-5 に絞る
        reranker = CohereRerank(
            model="rerank-multilingual-v3.0",
            top_n=5,
        )
        rerank_retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=base_retriever,
        )

        results_rerank = rerank_retriever.invoke(query)

        print("=" * 60)
        print(f"Rerank 結果 (TOP-5)")
        print("=" * 60)
        for i, d in enumerate(results_rerank, 1):
            score = d.metadata.get("relevance_score", "N/A")
            print(f"  [{i}] score={score:.4f}  {d.metadata.get('source')}")
            print(f"      {d.page_content[:100]}...")
            print()

        RERANK_AVAILABLE = True
    except Exception as e:
        print(f"⚠️ Reranking エラー: {e}")
        RERANK_AVAILABLE = False
else:
    print("ℹ️  COHERE_API_KEY 未設定のためスキップ")
    print("   リランキングを試すには Colab シークレットに COHERE_API_KEY を追加してください")
    RERANK_AVAILABLE = False


ℹ️  COHERE_API_KEY 未設定のためスキップ
   リランキングを試すには Colab シークレットに COHERE_API_KEY を追加してください


---
## 09. 検索方式 ④ — Multi-Query Retriever (クエリ展開)

LLM にユーザーのクエリを **複数の言い換え** に展開させ、それぞれで検索して結果を統合します。

- 例: 「Cloud-IAM のバックアップ方法は?」→ 「バックアップ戦略」「データ復旧手順」「バックアップ設定」...
- **曖昧・短いクエリ** に特に有効


In [19]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI
import logging

# LLM (クエリ展開用)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Multi-Query Retriever
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 4}),
    llm=llm,
)

# ログを有効化して生成されたクエリを確認
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("langchain.retrievers.multi_query")
logger.setLevel(logging.INFO)

results_mq = multiquery_retriever.invoke(query)
print(f"\nMulti-Query 結果: {len(results_mq)} 件 (重複排除済み)")
for i, d in enumerate(results_mq[:5], 1):
    print(f"  [{i}] {d.metadata.get('source')}")
    print(f"      {d.page_content[:100]}...")



Multi-Query 結果: 8 件 (重複排除済み)
  [1] https://documentation.cloud-iam.com/references/architecture-insights.html
      Cloud-IAM Architecture Insights | Cloud-IAM | DOCS...
  [2] https://documentation.cloud-iam.com/references/incident-management.html
      Cloud-IAM Incident Management | Cloud-IAM | DOCS...
  [3] https://documentation.cloud-iam.com/how-to-guides/cloud-iam-api.html
      Cloud-IAM REST API Automation & Integration | Cloud-IAM | DOCS...
  [4] https://documentation.cloud-iam.com/how-to-guides/extensions-management.html
      Extensions Management | Cloud-IAM | DOCS...
  [5] https://documentation.cloud-iam.com/references/deployment-maintenance.html
      Deployment Maintenance | Cloud-IAM | DOCS...


---
## 10. 検索方式 ⑤ — Parent Document Retriever (親子チャンク)

**小さいチャンクで精密に検索し、LLM には親の大きなチャンクを渡す** 戦略。

- 検索精度 (小チャンク) と 文脈の豊富さ (大チャンク) を **両立**
- 長文の文脈が必要な Q&A に有効


In [31]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_chroma import Chroma as ChromaParent

# 親チャンク: 大きめ (2000 文字)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
)

# 子チャンク: 小さめ (400 文字) — 検索用
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

# 子チャンク用の Vector Store (別インスタンス)
child_db = ChromaParent(
    collection_name="child_chunks",
    embedding_function=embeddings,
)

# 親チャンクを保存する DocStore
docstore = InMemoryStore()

# Parent Document Retriever を構築
parent_retriever = ParentDocumentRetriever(
    vectorstore=child_db,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# ドキュメントを登録 (子チャンクが Embedding → Vector Store、親チャンクが DocStore)
parent_retriever.add_documents(raw_docs)

print(f"子チャンク (Vector Store): {child_db._collection.count()} 件")
print(f"親チャンク (DocStore): {len(list(docstore.yield_keys()))} 件")


子チャンク (Vector Store): 4862 件
親チャンク (DocStore): 551 件


In [32]:
# Parent Document Retriever で検索
results_parent = parent_retriever.invoke(query)

print("=" * 60)
print(f"Parent Document Retriever 結果: {len(results_parent)} 件")
print("=" * 60)
for i, d in enumerate(results_parent, 1):
    print(f"  [{i}] source={d.metadata.get('source')}")
    print(f"      チャンクサイズ: {len(d.page_content)} 文字")
    print(f"      先頭 150 文字: {d.page_content[:150]}...")
    print()

# 通常の Retriever と比較
results_normal = vector_retriever.invoke(query)
avg_parent = sum(len(d.page_content) for d in results_parent) / max(len(results_parent), 1)
avg_normal = sum(len(d.page_content) for d in results_normal) / max(len(results_normal), 1)
print(f"平均チャンクサイズ比較:")
print(f"  通常 Retriever:         {avg_normal:.0f} 文字")
print(f"  Parent Doc Retriever:   {avg_parent:.0f} 文字")
print(f"  → Parent は約 {avg_parent / max(avg_normal, 1):.1f} 倍の文脈を LLM に渡せる")


Parent Document Retriever 結果: 2 件
  [1] source=https://documentation.cloud-iam.com/faq/users-management.html
      チャンクサイズ: 1998 文字
      先頭 150 文字: OTPMagic linksIdentity Provider (OIDC)SAML Clock SkewMulti-factor authenticationKeycloak CustomizationCustom ExtensionsKeycloak ThemeKeycloak Identity...

  [2] source=https://documentation.cloud-iam.com/faq/users-management.html
      チャンクサイズ: 1999 文字
      先頭 150 文字: Cloud-IAM plan accommodates, you have a couple of options to address the situation:1. Evaluate the Recommended Plan:Before onboarding to Cloud-IAM, ca...

平均チャンクサイズ比較:
  通常 Retriever:         449 文字
  Parent Doc Retriever:   1998 文字
  → Parent は約 4.4 倍の文脈を LLM に渡せる


---
## 11. 基本 RAG チェーン — LCEL パイプライン

ここまでで用意した Retriever を使って、RAG の完全なチェーンを構築します。

### LCEL パイプラインの処理フロー
```
入力クエリ
  ↓
① {"context": retriever, "question": RunnablePassthrough()}
     retriever → 関連チャンクを検索
     RunnablePassthrough → クエリをそのまま通す
  ↓
② prompt — テンプレートに context と question を挿入
  ↓
③ model — LLM で回答生成
  ↓
④ StrOutputParser — AIMessage → 文字列に変換
  ↓
出力
```


In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# プロンプトテンプレート
prompt = ChatPromptTemplate.from_template("""\
あなたは Cloud-IAM (Managed Keycloak サービス) の専門アシスタントです。

# 指示
- 以下の【文脈】だけを根拠に質問に答えてください
- 文脈に答えが含まれない場合は「提供された情報からは回答できません」と述べてください
- 回答は日本語で、簡潔かつ丁寧に

# 文脈
{context}

# 質問
{question}
""")

# LLM
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG チェーン (ハイブリッド検索を使用)
rag_chain = (
    {"context": hybrid_retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# 実行
output = rag_chain.invoke(query)
print("=" * 60)
print("RAG 回答 (Hybrid Retriever)")
print("=" * 60)
print(output)


RAG 回答 (Hybrid Retriever)
提供された情報からは回答できません。


---
## 12. 引用元付き RAG — RunnableParallel

シンプルなチェーンでは、検索結果 (context) が回答生成後に失われます。
`RunnableParallel` + `.assign()` を使って **回答と参照ドキュメントの両方** を返します。

これにより RAG の大きな利点である **「根拠の追跡可能性」** が実現できます。


In [33]:
from langchain_core.runnables import RunnableParallel

# 回答生成部分
answer_chain = prompt | model | StrOutputParser()

# 回答 + 参照ドキュメントを同時に返すチェーン
rag_chain_with_sources = (
    RunnableParallel({"context": hybrid_retriever, "question": RunnablePassthrough()})
    .assign(answer=answer_chain)
)

result = rag_chain_with_sources.invoke(query)

# --- 回答 ---
print("=" * 60)
print("回答:")
print("=" * 60)
print(result["answer"])

# --- 引用元 ---
print("\n" + "=" * 60)
print("参照したドキュメント:")
print("=" * 60)
for i, d in enumerate(result["context"], 1):
    print(f"  [{i}] {d.metadata.get('source')}")
    print(f"      {d.page_content[:120]}...")


回答:
提供された情報からは回答できません。

参照したドキュメント:
  [1] https://documentation.cloud-iam.com/references/architecture-insights.html
      Cloud-IAM Architecture Insights | Cloud-IAM | DOCS...
  [2] https://documentation.cloud-iam.com/faq/users-management.html
      .000.Therefore, a plan that accommodates more than 49.000 users, such as the 50k Plan with up to 50k users available, wo...
  [3] https://documentation.cloud-iam.com/references/plans.html
      .Visit Official SitePlans ​At Cloud-IAM, our plans are tailored to your needs, based on the total number of users and re...
  [4] https://documentation.cloud-iam.com/faq/users-management.html
      FAQ | Users Management | Cloud-IAM | DOCS...
  [5] https://documentation.cloud-iam.com/references/security.html
      Security - Cloud-IAM | Cloud-IAM | DOCS...
  [6] https://documentation.cloud-iam.com/get-started/migrate-to-cloud-iam.html
      Migrate to Cloud-IAM | Cloud-IAM | DOCS...
  [7] https://documentation.cloud-iam.com/references/incident-manage

---
## 13. ストリーミング出力

`.stream()` を使うと、LLM からの回答をトークン単位で逐次出力できます。
ユーザー体験の向上 (待ち時間の体感短縮) に重要です。


In [ ]:
print("=" * 60)
print("ストリーミング出力:")
print("=" * 60)

for chunk in rag_chain.stream(query):
    print(chunk, end="", flush=True)

print()  # 改行


---
## 14. 検索方式の比較実験

ここまでに実装した複数の検索方式を **同じクエリで一斉テスト** し、結果を比較します。

比較する方式:
1. **Similarity** (基本)
2. **MMR** (多様性重視)
3. **Hybrid** (BM25 + Vector)
4. **Multi-Query** (クエリ展開)
5. **Parent Document** (親子チャンク)
6. **Rerank** (オプション)

各方式から取得したチャンクで RAG を実行し、回答の違いを観察します。


In [ ]:
# 各 Retriever をまとめる
retrievers = {
    "Similarity":      retriever_sim,
    "MMR":             retriever_mmr,
    "Hybrid":          hybrid_retriever,
    "Multi-Query":     multiquery_retriever,
    "Parent-Doc":      parent_retriever,
}

if RERANK_AVAILABLE:
    retrievers["Rerank"] = rerank_retriever

# 全方式で検索 + RAG 回答を生成
test_query = "Cloud-IAM のシステムアーキテクチャはどのような構成ですか?"

print("=" * 60)
print(f"クエリ: {test_query}")
print("=" * 60)

# ログを一時的に抑制
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.WARNING)

for name, ret in retrievers.items():
    print(f"\n--- {name} ---")

    # 検索
    try:
        search_results = ret.invoke(test_query)
    except Exception as e:
        print(f"  ⚠️ エラー: {e}")
        continue

    print(f"  検索結果: {len(search_results)} 件")
    print(f"  ソース: {[d.metadata.get('source', '?') for d in search_results[:4]]}")

    # RAG 回答
    chain = (
        {"context": ret, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    answer = chain.invoke(test_query)
    # 最初の 200 文字だけ表示 (比較しやすくする)
    print(f"  回答 (先頭 200 文字): {answer[:200]}...")


---
## 15. メタデータフィルタ — 検索スコープの制限

`search_kwargs` の `filter` で、特定のメタデータ条件を満たすチャンクだけを検索できます。
権限管理・テナント分離・スコープ限定に必須の機能です。


In [ ]:
# どんなソースがあるか確認
sources = set()
for d in docs[:100]:
    sources.add(d.metadata.get("source", "unknown"))
print(f"ソースの例 (先頭 100 チャンク): {sorted(sources)}")

# 特定のソースだけに絞って検索
if sources:
    target_source = sorted(sources)[0]
    filtered_retriever = db.as_retriever(
        search_kwargs={
            "k": 4,
            "filter": {"source": target_source},
        }
    )
    results_filtered = filtered_retriever.invoke(query)
    print(f"\n--- filter: source = '{target_source}' ---")
    print(f"結果: {len(results_filtered)} 件")
    for i, d in enumerate(results_filtered, 1):
        print(f"  [{i}] {d.metadata.get('source')}: {d.page_content[:80]}...")

    # フィルタなしと比較
    results_unfiltered = vector_retriever.invoke(query)
    print(f"\n--- フィルタなし ---")
    print(f"結果: {len(results_unfiltered)} 件")
    for i, d in enumerate(results_unfiltered, 1):
        print(f"  [{i}] {d.metadata.get('source')}: {d.page_content[:80]}...")


---
## 16. まとめ — 実務での選び方

### 各検索方式の位置づけ

| 段階 | 構成 | 用途 |
|---|---|---|
| **プロトタイプ** | Similarity Search (k=4) | 学習・デモ |
| **実用最小** | Hybrid (BM25 + Vector) + メタデータフィルタ | 社内 RAG |
| **本番標準** ⭐ | Hybrid → Rerank → LLM | 商用プロダクト |
| **精度最重視** | Multi-Query → Hybrid → Rerank → LLM | 法務・医療・金融 |

### このノートブックで試したことの振り返り

1. **Document Loader**: WebBaseLoader で Cloud-IAM ドキュメントサイトから取得
2. **Text Splitter**: RecursiveCharacterTextSplitter (標準) と MarkdownHeaderTextSplitter (構造対応) の 2 種類
3. **Embeddings**: OpenAI `text-embedding-3-small`
4. **Vector Store**: Chroma + 永続化
5. **検索 6 方式**: Similarity / MMR / Threshold / Hybrid / Multi-Query / Parent-Doc (+ Rerank)
6. **RAG チェーン**: 基本 LCEL パイプライン、引用元付き、ストリーミング
7. **メタデータフィルタ**: 検索スコープの制限

### 次のステップ
- 評価: **Ragas** / **LangSmith** で検索品質・回答品質を定量計測
- Agentic RAG: **LangGraph** の `StateGraph` で検索 → 評価 → 再検索のループ
- プロンプト改善: 引用番号の指示、回答フォーマットの指定
- 他のデータソース: PDF、Confluence、ServiceNow のナレッジ記事
- Cloud-IAM ドキュメントの定期的な再インデックスによる最新情報の反映

詳細は付属の `RAG_explanation.md` を参照してください。
